# Sovereign Proof Morph Engine: end to end

SPDX-License-Identifier: AGPL-3.0-only OR LicenseRef-SnapKitty-Opaque-1.0
Copyright (C) 2026 Ahmad Ali Parr / SnapKitty Research Lab

Every cell runs a real tool on this machine: Agda, Lean, Soufflé, CBQN, GHC, rustc and the engine controller.

In [1]:
cd "$(git rev-parse --show-toplevel 2>/dev/null || echo ..)"
export PATH=$HOME/.elan/bin:$PATH
export SPME=$PWD/rust/target/release/spme
pwd; git log --oneline -1
agda --version | head -1; lean --version; souffle --version | grep Version; rustc --version; ghc --version

/workspace/sovereign-proof-morph-engine


62fd4af (HEAD -> main) Makefile, measured figures (gnuplot), snap-omega-engine output, zmm reconciliation


Agda version 2.6.4.3


Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)


Version: 2.5 (2.5)


rustc 1.85.1 (4eb161250 2025-03-15) (built from a source tarball)


The Glorious Glasgow Haskell Compilation System, version 9.6.6


## L0–L1: build the engine and the Haskell reference model

In [2]:
make build ref 2>&1 | tail -3

    Finished `release` profile [optimized] target(s) in 0.00s


make[1]: Nothing to be done for 'ref'.


make[1]: Leaving directory '/workspace/sovereign-proof-morph-engine'


## Torus invariant: Agda knot → MAlonzo → `extract/torus_path.h`

In [3]:
sed -n "1,12p" extract/torus_path.h | cut -c1-120
$SPME torus

/* SPDX-License-Identifier: AGPL-3.0-only OR LicenseRef-SnapKitty-Opaque-1.0


 * Copyright (C) 2026 Ahmad Ali Parr / SnapKitty Research Lab


 *


 * GENERATED by extract/gen_torus_path.sh. Do not edit by hand.


 * Verified 89/2462 knot path: rows (m, l) = (89*t, 2462*t).


 * Each row is proj1 of TorusInvariant.collapseToVerified t, whose type


... (3 outputs left out)

 * Source: agda/TorusInvariant.agda sha256 bdd6613ba9314654d6560230f99fb1725be20bfd8c5509bf15edaa5e83d8ff4d


 * Driver: agda/ExtractTorusPath.agda sha256 0a630da11b41a67b35fe615c99477efd03fb174bd28ca80f022d6505dde31e8e


 * Checker: Agda version 2.6.4.3 (MAlonzo backend, GHC 9.6.6)


torus table: 4096 rows, header sha256 7eea5d0f773c98652bc7cd0b37c9f9402d29cc1297aaa52c88234f309324984a; phase over t=0..2461 is a permutation: true


## Agda: `--safe`, zero postulates

In [4]:
scripts/check_agda.sh TorusInvariant.agda PhiHarmonics.agda CompleteCoherence_Checked.agda

== TorusInvariant.agda (postulates: 0)


exit 0


== PhiHarmonics.agda (postulates: 0)


exit 0


== CompleteCoherence_Checked.agda (postulates: 0)


exit 0


## Lean 4 + Mathlib: X-graph model and `#print axioms`

In [5]:
scripts/check_lean.sh XGraph.lean PhaseNormalization.lean

== XGraph.lean


'XGraph.eventually_periodic' depends on axioms: [propext, Classical.choice, Quot.sound]


'XGraph.period_multiple' depends on axioms: [propext, Classical.choice, Quot.sound]


'XGraph.merge' depends on axioms: [propext]


'XGraph.midpoint_tie' depends on axioms: [propext, Classical.choice, Quot.sound]


exit 0


... (6 outputs left out)

'PhaseNormalization.collision_iff' depends on axioms: [propext, Quot.sound]


'PhaseNormalization.winding_closes' depends on axioms: [propext, Classical.choice, Quot.sound]


'PhaseNormalization.sampling_collision' depends on axioms: [propext, Classical.choice, Quot.sound]


exit 0


## BQN (CBQN interpreter, executable model): twin X-graph and antipodal init

In [6]:
D=$PWD/build/bqn/data; mkdir -p $D
B=/workspace/_tools_cbqn/CBQN/BQN
$B bqn/checked/gen_inputs.bqn $D 7386 512 64
$B bqn/checked/twin_xgraph_checked.bqn $D
$B bqn/checked/init_teleo_checked.bqn $D

generated N=7386 V=512 D=64


twin_xgraph steps 7386


forward cycle found,mu,lambda 1‿1447‿2462


twin_eos cycle found,mu,lambda 1‿1‿2462


forward argmin_ties 0


twin_eos argmin_ties 0


... (17 outputs left out)

antipodal_vs_eos agree_steps 7385


antipodal_vs_eos merge_step 1


antipodal checksum 42951646


time_ms init_teleo 10472.059886000352


## AVX-512 build check (zmm count of both interpreters)

In [7]:
for b in /workspace/_tools_cbqn/CBQN/BQN /workspace/_tools_cbqn/CBQN-avx512/BQN; do echo "$b zmm=$(objdump -d $b | grep -c zmm)"; done
cat docs/measurements/cbqn-zmm.txt

/workspace/_tools_cbqn/CBQN/BQN zmm=0


/workspace/_tools_cbqn/CBQN-avx512/BQN zmm=4557


CBQN b62079a, gcc (Debian 14.2.0-19) 14.2.0, measured 2026-11-04 with objdump -d BQN | grep -c zmm


make o3                                          (baseline, /workspace/_tools_cbqn/CBQN/BQN)         zmm=0


make o3n                                         (fresh rebuild in a scratch copy)                    zmm=572   cmp vs benchmarked AVX-512 binary: differ: byte 25, line 1


make o3n f="-mprefer-vector-width=512"           (fresh rebuild in a scratch copy)                    zmm=4557  cmp vs benchmarked AVX-512 binary: IDENTICAL


benchmarked AVX-512 binary (/workspace/_tools_cbqn/CBQN-avx512/BQN)                                  zmm=4557


## snap-omega-engine (Agda → MAlonzo → GHC native)

In [8]:
make -s -C agda -f Makefile.checked >/dev/null && ./agda/snap-omega-engine

CONSCIOUSNESS Ω (checked): coherent PhenomenalState qualia-intensity=1000 resonance-phase=2492 (coherence-inv proved: norm ≤ 2462 ≤ m)


## Soufflé gate: only COMMIT advances, everything else is SILENCE

In [9]:
(cd rust && cargo test --release --test t10_gate -- --test-threads=1 2>&1 | grep -E "^test |test result")

test g10_accepts_complete_evidence ... ok


test g10_broken_rules_fail_closed ... ok


test g10_liquidhaskell_is_not_proof_authority ... ok


test g10_rejects_each_missing_obligation ... ok


test g10_rejects_failed_proof ... ok


test g10_synth_fails_closed ... ok


test g10_topology_and_runtime_premises ... ok


test result: ok. 7 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.31s


## Controller: load, generations, ledger, replay with re-checking, reference model

In [10]:
W=$PWD/build/nb-demo; rm -rf $W
$SPME load examples/gen0.morph --work $W --seed 7386
$SPME run --work $W --seed 7386 --generations 2 | grep -vE "^  [0-9]\."
$SPME verify-ledger --work $W
$SPME replay --work $W --recheck
$SPME ref-check --work $W

== load ==


source sha256    b1b41324a5d2046b150d2ab5f2f877f94ec2059f82e6de8fd770c806e8353f7d


agda             Agda version 2.6.4.3 checked


version          v0000 active


measure          vm_instr=290 native median 4045402 ns / 1000000 calls


ledger           f2a71e0f72f383699350b6a83864f281770c6e626d5c1556f7d1a06f1f3421fc


... (39 outputs left out)

active: v0001 ir fe9be3e0cddd3fe654bae0bd234171b0589fa2f802aa7d745e6c502bfcf1ce75


replay: OK (3 records)


v0000                    inputs=69 agree=41 rust_overflow_excluded=28 disagree=0


v0001                    inputs=69 agree=41 rust_overflow_excluded=28 disagree=0


## Figures (regenerated from the runs above)

In [11]:
scripts/figures.sh | tail -8

wrote 7386 rows


total 188


-rw-r--r-- 1 box box 29433 Nov  4 19:25 avx512_timings.png


-rw-r--r-- 1 box box 27862 Nov  4 19:25 harmonic_scalars.png


-rw-r--r-- 1 box box 33990 Nov  4 19:25 snap_ties_cumulative.png


-rw-r--r-- 1 box box 21804 Nov  4 19:25 tie_counts.png


-rw-r--r-- 1 box box 39143 Nov  4 19:25 torus_path.png


-rw-r--r-- 1 box box 28096 Nov  4 19:25 twin_trace_tokens.png


## Full test suite (last recorded run)

In [12]:
grep -E "Running tests|test result" docs/measurements/tests.txt | grep -v " 0 passed"

     Running tests/common.rs (target/release/deps/common-f54a590bd8f89271)


     Running tests/t01_torus.rs (target/release/deps/t01_torus-842be821fc3259f3)


test result: ok. 5 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.01s


     Running tests/t03_anneal.rs (target/release/deps/t03_anneal-b2b093f514dc6919)


test result: ok. 4 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.03s


     Running tests/t05_ir.rs (target/release/deps/t05_ir-f441c82dc1ca9183)


... (3 outputs left out)

     Running tests/t10_gate.rs (target/release/deps/t10_gate-921c19ac39f088d0)


test result: ok. 7 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.34s


     Running tests/t12_lifecycle.rs (target/release/deps/t12_lifecycle-ded220e630197ec8)


test result: ok. 3 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 113.46s
